# Context-aware Standard QML vs Context-aware PAQFE-v2 — Repeated 5-Fold GroupKFold

This notebook trains **only the two context-aware variants** for five shuffled composition-level GroupKFold repeats (seeds 42–46). It follows the reference `PAQFE_v2_Preliminary_5Fold_GroupKFold.ipynb` for dataset cleaning, composition grouping, 10-qubit / 3-layer quantum backbone, fold-only scaling, 150 epochs, and base learning rates.

**Important architecture note:** the context-aware variants retain the existing context-aware gain mechanism (`physics_gain` and `context_gain`). Context-aware PAQFE-v2 also retains the reference PAQFE-v2 physics-aware CNOT links and trainable `alpha` scaling with prior regularization. No baseline model runs are included.

## Fair-comparison alignment
This version aligns the context-aware grouped experiment with the observation-level notebook on the training budget (300 epochs), context-gain learning rate (LR_VQC = 0.02), and seed policy (init_seed = seed). The GroupKFold split strategy and train-fold-only scaling are preserved. PAQFE-v2 feature-wise alpha still uses LR_ALPHA = 0.005, matching the PAQFE-v2 baseline design. Results from this notebook should be compared against observation-level results only with the evaluation-protocol distinction clearly stated.


In [ ]:
import os, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from google.colab import files, drive
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fair-comparison settings aligned with the observation-level context-aware notebook
SEEDS = [42, 43, 44, 45, 46]
N_SPLITS = 5
EPOCHS = 300  # match observation-level context-aware experiment
N_QUBITS = 10
N_LAYERS = 3
LR_VQC = 0.02
LR_ALPHA = 0.005  # PAQFE feature-wise alpha only; retained from PAQFE-v2 baseline
LAMBDA_PRIOR = 0.01
TARGET = "UTS_MPa"

STANDARD_FEATURE_ORDER = [
    "xAl", "VEC", "dHmix", "dSmix", "delta", "Omega",
    "dchi", "Mean_Tm", "Temperature_C", "log10_Strain_rate"
]
PAQFE_FEATURE_ORDER = [
    "xAl", "VEC", "dchi", "dHmix", "dSmix", "Omega",
    "Mean_Tm", "delta", "Temperature_C", "log10_Strain_rate"
]
PHYSICS_FEATURES = [
    "xAl", "VEC", "dHmix", "dSmix", "delta", "Omega", "dchi", "Mean_Tm"
]
CONTEXT_FEATURES = ["Temperature_C", "log10_Strain_rate"]
COMPOSITION_COLS = ["Al_norm", "Co_norm", "Cr_norm", "Fe_norm", "Ni_norm"]
STRONG_RELATIONSHIP_DESCRIPTORS = {"delta", "Omega", "xAl", "VEC"}
MODELS = ["Context-aware Standard QML", "Context-aware PAQFE-v2"]

print("Torch:", torch.__version__, "| PennyLane:", qml.__version__)
print("Planned model-fold runs:", len(SEEDS) * N_SPLITS * len(MODELS))

## 1. Load the same Excel workbook (`B_Model_Ready`) and construct composition groups

In [ ]:
drive.mount("/content/drive")
RESULT_DIR = "/content/drive/MyDrive/PAQFE_ContextAware_Repeated_GroupKFold_5Seeds"
os.makedirs(RESULT_DIR, exist_ok=True)

FOLD_PATH = os.path.join(RESULT_DIR, "fold_metrics.csv")
OOF_PATH = os.path.join(RESULT_DIR, "oof_predictions.csv")
POOLED_PATH = os.path.join(RESULT_DIR, "pooled_oof_metrics_by_seed.csv")
SUMMARY_PATH = os.path.join(RESULT_DIR, "summary_metrics.csv")
SPLIT_PATH = os.path.join(RESULT_DIR, "fold_composition_audit.csv")

uploaded = files.upload()
xlsx_path = next(iter(uploaded.keys()))
df = pd.read_excel(xlsx_path, sheet_name="B_Model_Ready")

required = STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[required] = df[required].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=required).reset_index(drop=True).copy()
df["Composition_Group"] = df[COMPOSITION_COLS].round(3).astype(str).agg("_".join, axis=1)

print("Observations:", len(df))
print("Unique compositions:", df["Composition_Group"].nunique())
assert len(df) == 49, f"Expected 49 observations; found {len(df)}. Check workbook/cleaning."
assert df["Composition_Group"].nunique() == 14, (
    f"Expected 14 unique compositions; found {df['Composition_Group'].nunique()}."
)

## 2. Physics prior and physics-aware entanglement (PAQFE-v2 variant only)

In [ ]:
def ring_entangle(wires):
    if len(wires) < 2:
        return
    for i in range(len(wires)):
        qml.CNOT(wires=[wires[i], wires[(i + 1) % len(wires)]])

def physics_entangling_layer():
    # Exact grouped feature ordering used by the reference PAQFE-v2 notebook
    qml_feature_map = {f: i for i, f in enumerate(PAQFE_FEATURE_ORDER)}
    qml_group_blocks = {
        "composition": [qml_feature_map[f] for f in ["xAl", "VEC", "dchi"]],
        "thermodynamic": [qml_feature_map[f] for f in ["dHmix", "dSmix", "Omega", "Mean_Tm"]],
        "context": [qml_feature_map[f] for f in CONTEXT_FEATURES],
    }
    for wires in qml_group_blocks.values():
        ring_entangle(wires)
    qml.CNOT(wires=[qml_feature_map["delta"], qml_feature_map["Omega"]])
    qml.CNOT(wires=[qml_feature_map["xAl"], qml_feature_map["VEC"]])

physics_prior = torch.tensor(
    [1.2 if f in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0 for f in PAQFE_FEATURE_ORDER],
    dtype=torch.float64
)

## 3. Context-aware model builders

In [ ]:
def build_model(model_name, feature_names, init_seed):
    feature_names = list(feature_names)
    n_qubits = len(feature_names)
    idx = {f: i for i, f in enumerate(feature_names)}
    dev = qml.device("default.qubit", wires=n_qubits)
    use_paqfe = model_name == "Context-aware PAQFE-v2"

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        if use_paqfe:
            # Context-aware PAQFE-v2: retain PAQFE-v2 grouped CNOT links.
            groups = [
                [f for f in ["xAl", "VEC", "dchi"] if f in idx],
                [f for f in ["dHmix", "dSmix", "Omega", "Mean_Tm"] if f in idx],
                [f for f in ["delta"] if f in idx],
                [f for f in CONTEXT_FEATURES if f in idx],
            ]
            for group in groups:
                wires = [idx[f] for f in group]
                if len(wires) >= 2:
                    for j in range(len(wires)):
                        qml.CNOT(wires=[wires[j], wires[(j + 1) % len(wires)]])
            if "delta" in idx and "Omega" in idx:
                qml.CNOT(wires=[idx["delta"], idx["Omega"]])
            if "xAl" in idx and "VEC" in idx:
                qml.CNOT(wires=[idx["xAl"], idx["VEC"]])
        qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    weight_shape = qml.StronglyEntanglingLayers.shape(
        n_layers=N_LAYERS, n_wires=n_qubits
    )

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            torch.manual_seed(init_seed)
            np.random.seed(init_seed)
            self.weights = nn.Parameter(
                0.01 * torch.randn(weight_shape, dtype=torch.float64)
            )
            self.head = nn.Linear(n_qubits, 1, dtype=torch.float64)
            if use_paqfe:
                prior = torch.tensor(
                    [1.2 if f in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0
                     for f in feature_names],
                    dtype=torch.float64
                )
                self.register_buffer("physics_prior", prior)
                self.alpha = nn.Parameter(prior.clone())
            # Shared context-aware mechanism: trainable group gains.
            self.log_physics_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
            self.log_context_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))

        def forward(self, x):
            if use_paqfe:
                x = x * self.alpha
            gains = torch.ones(x.shape[1], dtype=x.dtype, device=x.device)
            physics_ids = [idx[f] for f in PHYSICS_FEATURES if f in idx]
            context_ids = [idx[f] for f in CONTEXT_FEATURES if f in idx]
            gains[physics_ids] = torch.exp(self.log_physics_gain)
            gains[context_ids] = torch.exp(self.log_context_gain)
            x = x * gains
            q = torch.stack(circuit(x, self.weights), dim=1)
            return self.head(q).squeeze(-1)

    return Model()

## 4. Fold training, metrics, and out-of-fold predictions

In [ ]:
def run_one(seed, fold, model_name, train_idx, test_idx):
    feature_names = (
        STANDARD_FEATURE_ORDER
        if model_name == "Context-aware Standard QML"
        else PAQFE_FEATURE_ORDER
    )
    Xtr_raw = df.iloc[train_idx][feature_names].to_numpy(dtype=np.float64)
    Xte_raw = df.iloc[test_idx][feature_names].to_numpy(dtype=np.float64)
    ytr_raw = df.iloc[train_idx][TARGET].to_numpy(dtype=np.float64)
    yte_raw = df.iloc[test_idx][TARGET].to_numpy(dtype=np.float64)

    # Fit both scalers on training data only, as in the reference notebook.
    xs = StandardScaler()
    Xtr = xs.fit_transform(Xtr_raw)
    Xte = xs.transform(Xte_raw)
    ys = StandardScaler()
    ytr = ys.fit_transform(ytr_raw.reshape(-1, 1)).ravel()

    # Match observation-level seed policy: same seed label across folds/models.
    # Models remain architecturally different; this standardizes the RNG policy.
    init_seed = seed
    torch.manual_seed(init_seed)
    np.random.seed(init_seed)
    random.seed(init_seed)

    model = build_model(model_name, feature_names, init_seed)
    param_groups = [
        {"params": [model.weights], "lr": LR_VQC},
        {"params": model.head.parameters(), "lr": LR_VQC},
        # Context-gain learning rate matches the observation-level notebook.
        {"params": [model.log_physics_gain, model.log_context_gain], "lr": LR_VQC},
    ]
    if model_name == "Context-aware PAQFE-v2":
        param_groups.append({"params": [model.alpha], "lr": LR_ALPHA})
    optimizer = torch.optim.Adam(param_groups)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float64)
    Xte_t = torch.tensor(Xte, dtype=torch.float64)
    ytr_t = torch.tensor(ytr, dtype=torch.float64)
    loss_fn = nn.MSELoss()
    model.train()

    for epoch in range(1, EPOCHS + 1):
        optimizer.zero_grad()
        pred = model(Xtr_t)
        loss = loss_fn(pred, ytr_t)
        if model_name == "Context-aware PAQFE-v2":
            loss = loss + LAMBDA_PRIOR * torch.mean(
                (model.alpha - model.physics_prior) ** 2
            )
        loss = loss + 0.01 * (
            model.log_physics_gain ** 2 + model.log_context_gain ** 2
        )
        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss: seed={seed}, fold={fold}, model={model_name}, epoch={epoch}"
            )
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                train_pred_scaled = model(Xtr_t).cpu().numpy()
            model.train()
            train_pred = ys.inverse_transform(
                train_pred_scaled.reshape(-1, 1)
            ).ravel()
            train_r2 = r2_score(ytr_raw, train_pred) if len(ytr_raw) > 1 else np.nan
            print(
                f"Seed {seed} | Fold {fold}/{N_SPLITS} | {model_name} | "
                f"Epoch {epoch:03d}/{EPOCHS} | "
                f"train MAE={mean_absolute_error(ytr_raw, train_pred):.2f} MPa | "
                f"RMSE={np.sqrt(mean_squared_error(ytr_raw, train_pred)):.2f} MPa | "
                f"R²={train_r2:.3f} | "
                f"physics gain={torch.exp(model.log_physics_gain).item():.3f} | "
                f"context gain={torch.exp(model.log_context_gain).item():.3f}",
                flush=True
            )

    model.eval()
    with torch.no_grad():
        pred_scaled = model(Xte_t).cpu().numpy()
    y_pred = ys.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    fold_row = {
        "Seed": seed, "Fold": fold, "Model": model_name, "N_Test": len(test_idx),
        "MAE": mean_absolute_error(yte_raw, y_pred),
        "RMSE": np.sqrt(mean_squared_error(yte_raw, y_pred)),
        "R2": r2_score(yte_raw, y_pred) if len(yte_raw) > 1 else np.nan,
    }
    oof_rows = [
        {"Seed": seed, "Fold": fold, "Model": model_name, "Row_Index": int(i),
         "Composition_Group": df.iloc[i]["Composition_Group"],
         "y_true": float(yt), "y_pred": float(yp)}
        for i, yt, yp in zip(test_idx, yte_raw, y_pred)
    ]
    return fold_row, oof_rows

## 5. Run five repeats × five folds (same folds for both models within each seed)

In [ ]:
fold_records = []
oof_records = []
split_records = []

for seed in SEEDS:
    splitter = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    splits = list(splitter.split(
        df[STANDARD_FEATURE_ORDER], df[TARGET], df["Composition_Group"]
    ))
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        train_groups = sorted(df.iloc[train_idx]["Composition_Group"].unique().tolist())
        test_groups = sorted(df.iloc[test_idx]["Composition_Group"].unique().tolist())
        overlap = set(train_groups).intersection(test_groups)
        if overlap:
            raise AssertionError(f"Composition leakage in seed={seed}, fold={fold}: {overlap}")
        split_records.append({
            "Seed": seed, "Fold": fold, "N_Train": len(train_idx), "N_Test": len(test_idx),
            "N_Train_Compositions": len(train_groups), "N_Test_Compositions": len(test_groups),
            "Train_Compositions": " | ".join(train_groups),
            "Test_Compositions": " | ".join(test_groups),
        })
        for model_name in MODELS:
            print(f"\nStarting seed={seed}, fold={fold}/{N_SPLITS}, model={model_name}")
            fold_row, oof_rows = run_one(seed, fold, model_name, train_idx, test_idx)
            fold_records.append(fold_row)
            oof_records.extend(oof_rows)
            pd.DataFrame(fold_records).to_csv(FOLD_PATH, index=False)
            pd.DataFrame(oof_records).to_csv(OOF_PATH, index=False)
            pd.DataFrame(split_records).to_csv(SPLIT_PATH, index=False)

fold_results = pd.DataFrame(fold_records)
oof_results = pd.DataFrame(oof_records)
print("\nAll runs complete:", len(fold_results), "model-fold runs")

## 6. Summaries: pooled OOF metrics per seed and fold-averaged metrics

In [ ]:
# Each seed should have one out-of-fold prediction for every observation and model.
dup = oof_results.duplicated(subset=["Seed", "Model", "Row_Index"], keep=False)
if dup.any():
    raise ValueError("Duplicate OOF predictions found.")
pooled_rows = []
for (seed, model_name), g in oof_results.groupby(["Seed", "Model"], sort=True):
    if len(g) != len(df):
        raise ValueError(
            f"Expected {len(df)} OOF rows for seed={seed}, model={model_name}; found {len(g)}"
        )
    yt, yp = g["y_true"].to_numpy(), g["y_pred"].to_numpy()
    pooled_rows.append({
        "Seed": int(seed), "Model": model_name, "N_OOF": len(g),
        "MAE": mean_absolute_error(yt, yp),
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "R2": r2_score(yt, yp),
    })

pooled_by_seed = pd.DataFrame(pooled_rows)
summary_metrics = pooled_by_seed.groupby("Model")[["MAE", "RMSE", "R2"]].agg(["mean", "std"]).round(3)
fold_summary = fold_results.groupby("Model")[["MAE", "RMSE", "R2"]].agg(["mean", "std"]).round(3)

display(pooled_by_seed.round(3))
display(summary_metrics)
display(fold_summary)

pooled_by_seed.to_csv(POOLED_PATH, index=False)
summary_metrics.to_csv(SUMMARY_PATH)
fold_summary.to_csv(os.path.join(RESULT_DIR, "fold_metrics_mean_sd.csv"))
fold_results.to_csv(FOLD_PATH, index=False)
oof_results.to_csv(OOF_PATH, index=False)

print("Results folder:", RESULT_DIR)
print("Saved: fold_metrics.csv, oof_predictions.csv, fold_composition_audit.csv,")
print("       pooled_oof_metrics_by_seed.csv, summary_metrics.csv, fold_metrics_mean_sd.csv")

## Interpretation reminder

- **Primary comparison:** pooled OOF MAE/RMSE/R² calculated separately for each seed, then summarized across seeds.
- **Fold-averaged metrics** are also reported for continuity with the preliminary notebook, but folds have unequal numbers of observations.
- Every observation is predicted out-of-fold once per seed. The composition-group audit is saved so the train/test separation can be checked.
- This notebook creates 50 training runs: 5 seeds × 5 folds × 2 context-aware models. It does not run the original non-context-aware baselines.

## Explainability — one composition-held-out fold per context-aware model

Retrains the selected seed/fold with the same architecture, training-only scalers and regularization as the main experiment. Feature importance is fold-specific; aggregate over folds/seeds for paper-level conclusions.

In [ ]:
EXPLAIN_SEED = SEEDS[0]
EXPLAIN_FOLD = 1
EXPLAIN_REPEATS = 30

def fit_context_group_fold_for_explainability(seed, fold, model_name, train_idx, test_idx):
    feature_names = STANDARD_FEATURE_ORDER if model_name == 'Context-aware Standard QML' else PAQFE_FEATURE_ORDER
    Xtr_raw=df.iloc[train_idx][feature_names].to_numpy(dtype=np.float64); Xte_raw=df.iloc[test_idx][feature_names].to_numpy(dtype=np.float64)
    ytr_raw=df.iloc[train_idx][TARGET].to_numpy(dtype=np.float64); yte_raw=df.iloc[test_idx][TARGET].to_numpy(dtype=np.float64)
    xs,ys=StandardScaler(),StandardScaler(); Xtr=xs.fit_transform(Xtr_raw); Xte=xs.transform(Xte_raw); ytr=ys.fit_transform(ytr_raw.reshape(-1,1)).ravel()
    init_seed=seed; torch.manual_seed(init_seed); np.random.seed(init_seed); random.seed(init_seed)
    model=build_model(model_name,feature_names,init_seed)
    groups=[{'params':[model.weights],'lr':LR_VQC},{'params':model.head.parameters(),'lr':LR_VQC},{'params':[model.log_physics_gain,model.log_context_gain],'lr':LR_VQC}]
    if model_name == 'Context-aware PAQFE-v2': groups.append({'params':[model.alpha],'lr':LR_ALPHA})
    opt=torch.optim.Adam(groups); Xt=torch.tensor(Xtr,dtype=torch.float64); yt=torch.tensor(ytr,dtype=torch.float64); loss_fn=nn.MSELoss(); model.train()
    for _ in range(EPOCHS):
        opt.zero_grad(); loss=loss_fn(model(Xt),yt)
        if model_name == 'Context-aware PAQFE-v2': loss=loss+LAMBDA_PRIOR*torch.mean((model.alpha-model.physics_prior)**2)
        loss=loss+0.01*(model.log_physics_gain**2+model.log_context_gain**2)
        loss.backward(); opt.step()
    return model,Xte,yte_raw,ys,feature_names

def context_group_permutation_importance(model,Xtest,ytrue,ys,feature_names,repeats=30,random_state=42):
    rng=np.random.default_rng(random_state); model.eval()
    def pred(X):
        with torch.no_grad(): z=model(torch.tensor(X,dtype=torch.float64)).cpu().numpy().reshape(-1,1)
        return ys.inverse_transform(z).ravel()
    baseline=mean_absolute_error(ytrue,pred(Xtest)); rows=[]
    for j,name in enumerate(feature_names):
        vals=[]
        for _ in range(repeats):
            Xp=Xtest.copy(); Xp[:,j]=rng.permutation(Xp[:,j]); vals.append(mean_absolute_error(ytrue,pred(Xp))-baseline)
        rows.append({'Feature':name,'MAE_increase_mean':np.mean(vals),'MAE_increase_sd':np.std(vals,ddof=1)})
    return pd.DataFrame(rows).sort_values('MAE_increase_mean',ascending=False)

splitter=GroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=EXPLAIN_SEED)
splits=list(splitter.split(df[STANDARD_FEATURE_ORDER],df[TARGET],df['Composition_Group']))
train_idx,test_idx=splits[EXPLAIN_FOLD-1]
assert not (set(df.iloc[train_idx]['Composition_Group']) & set(df.iloc[test_idx]['Composition_Group']))
for model_name in MODELS:
    model,Xte,yte,ys,feature_names=fit_context_group_fold_for_explainability(EXPLAIN_SEED,EXPLAIN_FOLD,model_name,train_idx,test_idx)
    imp=context_group_permutation_importance(model,Xte,yte,ys,feature_names,repeats=EXPLAIN_REPEATS,random_state=EXPLAIN_SEED)
    print(model_name,'seed',EXPLAIN_SEED,'fold',EXPLAIN_FOLD); display(imp)
    imp.to_csv(model_name.replace(' ','_').replace('-','_')+f'_seed{EXPLAIN_SEED}_fold{EXPLAIN_FOLD}_permutation_importance.csv',index=False)
    print('Physics gain:',float(torch.exp(model.log_physics_gain).detach()),'Context gain:',float(torch.exp(model.log_context_gain).detach()))
    if hasattr(model,'alpha'):
        prior=model.physics_prior.detach().cpu().numpy(); alpha=model.alpha.detach().cpu().numpy()
        alpha_df=pd.DataFrame({'Feature':feature_names,'Physics_prior':prior,'Learned_alpha':alpha,'Change':alpha-prior}); display(alpha_df)
        alpha_df.to_csv(model_name.replace(' ','_').replace('-','_')+f'_seed{EXPLAIN_SEED}_fold{EXPLAIN_FOLD}_alpha.csv',index=False)
